# Investigação forense da apuração presidencial — 2026

## O que os dados mostram? Existe evidência de fraude?

**Um laboratório de investigação aberto ao público, escrito para ser compreendido sem conhecimento de programação.**

A pergunta não é se uma curva *parece estranha*. Queremos descobrir se há evidências verificáveis de **alteração, acréscimo, omissão ou contabilização indevida de votos**.

Para isso, vamos separar três coisas que frequentemente são misturadas:

1. **Votos registrados nas urnas** — boletins individuais (BU/BWEB).
2. **Horário em que esses registros aparecem como recebidos** — `DT_BU_RECEBIDO`.
3. **Horário em que os totais aparecem publicados ao público** — série nacional EA20.

Uma diferença entre (2) e (3) pode significar processamento, fila de divulgação, diferenças de relógio ou algo que exige investigação. **Não é, por si só, uma alteração de votos.**

Este painel usa resultados locais que já passaram por verificações de integridade no projeto TSE. Não inventa dados, não procura apenas resultados favoráveis à hipótese de fraude e não confunde compatibilidade matemática com comprovação de um mecanismo real.

**Como ler:** cada figura traz uma pergunta popular, a observação, a explicação alternativa e o que ainda precisamos conferir. Os gráficos são exportados como PNG para compartilhar, acompanhados das ressalvas correspondentes.

*Escopo: eleição presidencial, primeiro turno de 4 de outubro de 2026. Análise independente. Nem o TSE nem terceiros endossam automaticamente estas interpretações.*

## Perguntas e critérios de prova

| Pergunta do público | Evidência que realmente a responderia |
|---|---|
| **Os votos foram alterados?** | Divergência reproduzível, por seção e candidato, entre BU autenticado e total oficial correspondente |
| **Votos foram adicionados ou omitidos?** | Contagens por seção que não reconciliam com o conjunto correto de BUs, após esclarecer eleições, locais no exterior e votos anulados |
| **A divulgação foi artificialmente controlada?** | Contradição documentada entre horários, eventos e registros operacionais, com relógios e semânticas certificados |
| **A linha reta da apuração prova fraude?** | Não. Precisamos de testes contra mecanismos normais e de evidência adicional que diferencie as hipóteses |

**Procurar fraude não significa assumir que houve fraude.** Significa formular testes capazes de detectar adulteração caso ela exista — e admitir quando os dados disponíveis não conseguem decidir.

## 1 — Carregar as evidências e conferir sua integridade

Os gráficos a seguir são gerados a partir de **CSV e JSON já produzidos no seu próprio computador** pelas fases V, W, X e Y. Arquivos inexistentes serão marcados como pendentes; nenhum valor será inventado. Cada CSV de saída é novamente verificado por SHA-256.

In [ ]:
from pathlib import Path
import sys, json, hashlib, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def locate_repo():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p/"start_windows.bat").exists():
            return p
    raise FileNotFoundError("Abra o painel no Jupyter iniciado na pasta TSE.")

REPO=locate_repo()
sys.path.insert(0,str(REPO))
from urnas import root_path
BASE=root_path(None)/"analysis"/"periodicidade_bweb_2026"
EXPORT=BASE/"divulgacao_publica_2026"
EXPORT.mkdir(parents=True,exist_ok=True)

NAVY="#183153"
TEAL="#197A74"
AMBER="#CF8540"
RED="#B94F4D"
GREY="#6B7280"
LIGHT="#E6EDF2"
plt.rcParams.update({
    "figure.figsize":(11,5),"figure.dpi":115,
    "savefig.dpi":190,
    "font.size":11,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.labelcolor":NAVY,"text.color":NAVY,
    "axes.titleweight":"bold",
})

def digest(path):
    sha=hashlib.sha256()
    with path.open("rb") as fp:
        for chunk in iter(lambda:fp.read(4*1024*1024),b""):
            sha.update(chunk)
    return sha.hexdigest()

def summary_if_available(filename,expected_status):
    path=BASE/filename
    if not path.is_file():
        display(Markdown("**Pendente:** relatório `"+filename+"` não foi gerado neste computador."))
        return None
    d=json.loads(path.read_text(encoding="utf-8"))
    if d.get("status")!=expected_status:
        raise RuntimeError("Status inesperado em "+filename)
    if d.get("year")!=2026 or d.get("round")!=1:
        raise RuntimeError("Arquivo de outra eleição/turno: "+filename)
    return d

def checked_csv(meta):
    path=Path(meta["path"])
    if not path.is_file():
        path=BASE/path.name
    if not path.is_file() or digest(path)!=meta["sha256"]:
        raise RuntimeError("SHA-256 não confere em "+str(path))
    return pd.read_csv(path,sep=";",encoding="utf-8-sig",
                       dtype={"UF":str,"municipio":str,"zona":str})

def publish(fig,name):
    fig.tight_layout()
    fig.savefig(EXPORT/(name+".png"),dpi=190,facecolor="white",bbox_inches="tight")
    plt.show()

t=summary_if_available("phase5p_holdout_geographic_curve_2026_summary.json",
                       "PREVISAO_BWEB_FORA_DA_AMOSTRA_1T_2026")
v=summary_if_available("phase5q_zone_holdout_2026_summary.json",
                       "ZONA_PRE_CORTE_TESTADA_NA_CAUDA_BWEB_2026")
w=summary_if_available("phase5r_cutoff_placebos_2026_summary.json",
                       "PLACEBOS_TEMPORAIS_INTRA_ZONA_2026")
x=summary_if_available("phase5s_ea20_publication_mechanics_2026_summary.json",
                       "MECANISMO_DA_CURVA_PUBLICADA_EA20_2026")
y=summary_if_available("phase5t_receipt_vs_publication_2026_summary.json",
                       "BWEB_RECEBIMENTO_VS_PUBLICACAO_EA20_2026")

if x:
    actual_ea20=REPO/"data/forensics/analysis/phase5a_presidential_national_2026.csv"
    if digest(actual_ea20)!=x["source_sha256"]["EA20_national"]:
        raise RuntimeError("SHA da série nacional EA20 não confere")
print("Pasta de figuras:",EXPORT)
print("Fases encontradas:",", ".join(n for n,d in [("T",t),("V",v),("W",w),("X",x),("Y",y)] if d) or "nenhuma")

## 2 — Por que a apuração parece uma linha reta?

**Pergunta:** uma linha acumulada muito lisa seria impossível sem interferência?

O gráfico de cima mostra a participação nacional **acumulada** de Lula na cauda após as 20h04. A linha tracejada é um experimento: mantém exatamente o volume de votos de cada atualização, mas dá a todos os lotes a mesma proporção que só conhecemos **ao final**.

**Importante:** esse experimento é retrospectivo; ele não pode ser apresentado como previsão independente. A escala vertical está ampliada para tornar visível uma variação pequena em pontos percentuais.

In [ ]:
if x:
    b=checked_csv(x["output_files"]["batches"])
    start=x["at_cutoff"]
    base_l=100*start["lula"]/start["valid"]
    b0=pd.DataFrame([{"cum_sections":start["sections"],"actual_cum_lula_pct":base_l,
                       "oracle_cum_lula_pct":base_l}])
    curve=pd.concat([b0,b[["cum_sections","actual_cum_lula_pct","oracle_cum_lula_pct"]]],
                    ignore_index=True)
    fig,ax=plt.subplots(figsize=(11,5))
    ax.plot(curve["cum_sections"],curve["actual_cum_lula_pct"],
            color=TEAL,lw=2.8,label="Apuração realmente publicada")
    ax.plot(curve["cum_sections"],curve["oracle_cum_lula_pct"],
            color=AMBER,lw=2.3,ls="--",
            label="Controle: proporção constante (conhecendo o final)")
    ax.set(title="Uma linha muito lisa também pode surgir da própria acumulação",
           xlabel="Seções contabilizadas na divulgação nacional (EA20)",
           ylabel="Participação acumulada de Lula (%)")
    ax.legend(loc="lower right",frameon=False)
    ax.grid(axis="y",alpha=.16)
    ax.text(.015,.025,"Eixo vertical ampliado • controle retrospectivo, não prova de mecanismo real",
            transform=ax.transAxes,color=GREY,fontsize=9)
    publish(fig,"01_linha_acumulada_e_controle")
    display(Markdown(
        "**O que observamos:** a curva publicada é muito regular, mas uma simulação "
        "que conserva os volumes publicados produz outra ainda mais regular. "
        "**O que isso significa:** a aparência de linha reta, sozinha, não permite "
        "identificar adulteração. **O que falta:** confirmar como os BUs foram efetivamente "
        "agrupados e publicados pelo sistema."
    ))
else:
    display(Markdown("Execute a **opção X** para construir esta figura diretamente dos resultados verificados."))

## 3 — O que a linha reta esconde? Quatro grupos de votos diferentes

O resultado acumulado pode mudar pouco porque já há dezenas de milhões de votos no denominador. Por isso o comportamento dos **novos votos de cada lote** é mais revelador do que olhar apenas para a média acumulada.

Aqui os 18,8 milhões de votos publicados depois do corte são divididos em **quatro grupos consecutivos de tamanho aproximado**. Cada barra começa em zero e tem a mesma escala (0–100%); candidatos menores aparecem como "demais".

In [ ]:
if x:
    blocks=checked_csv(x["output_files"]["blocks"]).sort_values("block")
    labels=["Bloco "+str(int(z)) for z in blocks["block"]]
    lula=blocks["lula_pct"].to_numpy()
    flavio=blocks["flavio_pct"].to_numpy()
    others=100-lula-flavio
    idx=np.arange(len(blocks))
    fig,ax=plt.subplots(figsize=(11,5))
    ax.barh(idx,lula,color=TEAL,label="Lula")
    ax.barh(idx,flavio,left=lula,color=AMBER,label="Flávio Bolsonaro")
    ax.barh(idx,others,left=lula+flavio,color=LIGHT,label="Demais votos válidos")
    ax.set_yticks(idx,labels)
    ax.invert_yaxis()
    ax.set_xlim(0,100)
    ax.set(xlabel="% dos votos válidos acrescentados em cada bloco",
           title="A linha acumulada escondia mudanças reais na composição dos lotes")
    ax.legend(ncol=3,loc="lower center",bbox_to_anchor=(.5,-.32),frameon=False)
    for i,(l,valids) in enumerate(zip(lula,blocks["valid"])):
        ax.text(max(5,l/2),i,f"{l:.1f}% Lula",va="center",ha="center",
                color="white",weight="bold")
        ax.text(101,i,f"{valids/1_000_000:.2f} mi de votos",va="center",
                fontsize=9,color=GREY)
    ax.set_xlim(0,119)
    publish(fig,"02_quatro_blocos_de_votos")
    display(Markdown("**O que observamos:** entre os quatro grupos de votos "
        "publicados, a participação de Lula muda consideravelmente. "
        "**Explicação alternativa:** regiões e seções com perfis diferentes "
        "podem chegar em horários diferentes. "
        "**Pergunta ainda aberta:** os BUs correspondentes explicam esses quatro blocos "
        "ou existe uma divergência de contabilização?"))
else:
    display(Markdown("Execute a **opção X** para gerar a comparação dos quatro blocos."))

## 4 — O mapa do Brasil: onde os votos chegaram mais tarde?

A primeira pergunta geográfica não é “qual estado tem mais votos para determinado candidato?”. É **em quais estados uma parcela maior dos votos das urnas foi recebida depois das 20h04min39s?**

Cada estado é colorido pela proporção de seus próprios votos válidos BWEB que têm `DT_BU_RECEBIDO` posterior ou igual ao corte. Assim, estados populosos não dominam simplesmente porque têm mais eleitores.

- **Cores fortes** significam uma fração maior dos votos daquele estado recebida depois do corte; **não significam fraude**.
- O mapa cobre as **27 UFs brasileiras**. Os votos do exterior (ZZ) são analisados separadamente e não são colocados artificialmente sobre o território.
- O mapa serve para revelar a **geografia da recepção**, não o momento da publicação nacional EA20.

Malha: limites estaduais simplificados, derivados de dados públicos do IBGE, licença CC0. É uma ilustração territorial, não uma base para medir distâncias ou limites com precisão cadastral.

In [ ]:
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable

GEO_PATH=REPO/"data"/"geografia"/"brasil_ufs_ibge_simplificado.geojson"
geo=json.loads(GEO_PATH.read_text(encoding="utf-8"))
BR_UFS=set("AC AL AM AP BA CE DF ES GO MA MG MS MT PA PB PE PI PR RJ RN RO RR RS SC SE SP TO".split())
features={f["properties"]["sigla"]:f for f in geo["features"]}
assert geo["type"]=="FeatureCollection" and set(features)==BR_UFS
REGIONS={
    "Norte":("AC","AP","AM","PA","RO","RR","TO"),
    "Nordeste":("AL","BA","CE","MA","PB","PE","PI","RN","SE"),
    "Centro-Oeste":("DF","GO","MT","MS"),
    "Sudeste":("ES","MG","RJ","SP"),
    "Sul":("PR","RS","SC"),
}
assert len({uf for members in REGIONS.values() for uf in members})==27
REGION_OF={uf:reg for reg,group in REGIONS.items() for uf in group}
STATE_LABELS={
    "AC":(-70.0,-9.2),"AM":(-64.5,-4.0),"PA":(-52.0,-4.6),
    "RO":(-63.5,-10.8),"RR":(-61.3,1.5),"TO":(-48.4,-10.4),
    "MA":(-45.5,-5.4),"PI":(-42.7,-7.7),"BA":(-41.5,-12.5),
    "CE":(-39.7,-5.0),"MT":(-56.0,-12.7),"GO":(-49.5,-16.0),
    "MS":(-54.3,-20.8),"MG":(-45.5,-18.5),"SP":(-49.1,-22.8),
    "PR":(-51.9,-24.9),"SC":(-50.7,-27.0),"RS":(-53.0,-30.2),
}

def paint_brazil(ax, metric, cmap, norm, title, show_labels=True):
    colors=plt.get_cmap(cmap)
    for uf,feature in features.items():
        item=metric.get(uf)
        finite=item is not None and np.isfinite(float(item))
        color=colors(norm(float(item))) if finite else "#E4E7EB"
        geom=feature["geometry"]
        multipolygons=([geom["coordinates"]] if geom["type"]=="Polygon"
                       else geom["coordinates"])
        for polygon in multipolygons:
            for k,ring in enumerate(polygon):
                coords=np.asarray(ring,dtype=float)
                # Interiores sao raros e omitidos; figura apenas ilustrativa.
                if k==0:
                    ax.fill(coords[:,0],coords[:,1],
                            facecolor=color,edgecolor="white",
                            linewidth=.6,zorder=2)
        if show_labels and uf in STATE_LABELS:
            px,py=STATE_LABELS[uf]
            ax.text(px,py,uf,fontsize=7,ha="center",va="center",
                    color=NAVY,weight="bold",zorder=3)
    ax.set_xlim(-74.5,-32.0)
    ax.set_ylim(-34.5,5.5)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title,loc="left",fontsize=14,pad=12)
    return ScalarMappable(norm=norm,cmap=colors)

if t:
    uf_t=checked_csv(t["output_files"]["per_UF"])
    uf_t=uf_t[uf_t["UF"].isin(BR_UFS)].copy()
    assert not uf_t["UF"].duplicated().any()
    assert set(uf_t["UF"])==BR_UFS, "As 27 UFs não estão representadas no modelo T"
    for field in ("pre_valid","post_valid"):
        uf_t[field]=pd.to_numeric(uf_t[field],errors="raise")
    uf_t["late_pct"]=100*uf_t["post_valid"]/(uf_t["pre_valid"]+uf_t["post_valid"])
    late_values=dict(zip(uf_t["UF"],uf_t["late_pct"]))
    lim=math.ceil(uf_t["late_pct"].max()/5)*5
    fig,ax=plt.subplots(figsize=(8,8))
    sm=paint_brazil(ax,late_values,"YlGnBu",
                    Normalize(vmin=0,vmax=max(5,lim)),
                    "Onde os votos chegaram mais tarde?")
    cbar=fig.colorbar(sm,ax=ax,orientation="horizontal",
                      shrink=.66,pad=.01,fraction=.035)
    cbar.set_label("% dos votos válidos da UF recebidos a partir de 20h04")
    fig.text(.1,.055,"Fonte: BU/BWEB, primeiro turno 2026  |  ZZ (exterior) fora do mapa",
             color=GREY,fontsize=9)
    publish(fig,"06_mapa_brasil_recebimento_tardio")
    region_t=(uf_t.assign(regiao=uf_t["UF"].map(REGION_OF))
            .groupby("regiao",as_index=False)[["pre_valid","post_valid"]].sum())
    region_t["late_pct"]=100*region_t["post_valid"]/(region_t["pre_valid"]+region_t["post_valid"])
    display(region_t.sort_values("late_pct",ascending=False).round(2).rename(columns={
        "regiao":"Região","pre_valid":"Votos antes","post_valid":"Votos depois",
        "late_pct":"% recebidos depois"}))
    display(Markdown("**O que observamos:** o mapa revela diferenças no ritmo "
        "de recebimento entre estados. **O que não demonstra:** que os votos foram "
        "alterados ou que a divulgação EA20 ocorreu na mesma ordem."))
else:
    display(Markdown("**Mapa pendente:** não encontrei o sumário verificado da fase T."))

## 5 — Mapa por estado e comparação por região: onde a mudança foi maior?

O segundo mapa responde a **uma pergunta diferente**: nas zonas eleitorais que já possuíam pelo menos 1.000 votos válidos recebidos antes **e** depois das 20h04, a participação de Lula aumentou ou diminuiu?

- Um valor de **+2 pontos percentuais** significa que, naquele grupo de zonas do estado, a participação de Lula entre os BUs posteriores foi em média 2 pontos maior do que entre os anteriores.
- É uma diferença **dentro das mesmas zonas**, mas as seções de uma zona não são necessariamente uma amostra aleatória de seus eleitores.
- A cor **não é um índice de fraude**; identifica um contraste temporal a investigar.
- A comparação regional pondera pelo **número de votos válidos posteriores nas zonas elegíveis**, e não pela área do mapa nem pelo número de estados.

Mostramos também o número de votos das zonas elegíveis: uma variação grande sobre uma amostra pequena não pode ser confundida com a situação de uma região inteira.

In [ ]:
if w:
    uf_w=checked_csv(w["output_files"]["ufs"])
    uf_w=uf_w[uf_w["UF"].isin(BR_UFS)].copy()
    assert not uf_w["UF"].duplicated().any()
    for field in ("weighted_shift_lula_pp","paired_post_valid",
                  "paired_pre_valid","paired_zones","positive_zones"):
        uf_w[field]=pd.to_numeric(uf_w[field],errors="raise")
    uf_w["regiao"]=uf_w["UF"].map(REGION_OF)
    assert uf_w["regiao"].notna().all()
    shift_values=dict(zip(uf_w["UF"],uf_w["weighted_shift_lula_pp"]))
    limit=max(1.0, float(uf_w["weighted_shift_lula_pp"].abs().max()))
    fig,ax=plt.subplots(figsize=(8,8))
    sm=paint_brazil(ax,shift_values,"PuOr",
                    TwoSlopeNorm(vmin=-limit,vcenter=0,vmax=limit),
                    "Dentro das zonas, quanto mudou?")
    cbar=fig.colorbar(sm,ax=ax,orientation="horizontal",
                      shrink=.7,pad=.01,fraction=.035)
    cbar.set_label("Lula: recebidos depois − antes (pontos percentuais)")
    fig.text(.1,.055,"Zonas com ≥1.000 válidos antes e depois  |  ZZ fora do mapa",
             color=GREY,fontsize=9)
    publish(fig,"07_mapa_brasil_mudanca_intrazona")
    group=[]
    for reg,table in uf_w.groupby("regiao"):
        weights=table["paired_post_valid"].to_numpy(dtype=float)
        rate=table["weighted_shift_lula_pp"].to_numpy(dtype=float)
        valid=float(weights.sum())
        if valid>0:
            group.append({"regiao":reg,"shift_pp":np.average(rate,weights=weights),
                          "valid":int(valid),"zones":int(table["paired_zones"].sum()),
                          "positive":int(table["positive_zones"].sum())})
    region_w=pd.DataFrame(group).sort_values("shift_pp")
    fig,ax=plt.subplots(figsize=(10,4.6))
    ax.barh(region_w["regiao"],region_w["shift_pp"],
            color=[TEAL if z>=0 else AMBER for z in region_w["shift_pp"]])
    ax.axvline(0,color=GREY,lw=1)
    ax.set(xlabel="Diferença de Lula nos BUs posteriores (pontos percentuais)",
           title="O contraste também pode ser visto nas cinco grandes regiões")
    ax.grid(axis="x",alpha=.15)
    xr=max(1,float(region_w["shift_pp"].abs().max()))
    ax.set_xlim(min(-xr*.45,float(region_w["shift_pp"].min())*1.2)-.15,
                max(xr*.9,float(region_w["shift_pp"].max())*1.2)+.25)
    for i,r in enumerate(region_w.itertuples()):
        offset=.03 if r.shift_pp>=0 else -.03
        ax.text(r.shift_pp+offset,i,
                f"{r.shift_pp:+.2f} pp".replace(".",","),
                va="center",ha="left" if r.shift_pp>=0 else "right",
                color=NAVY,fontsize=9)
    publish(fig,"08_regioes_brasil_mudanca_intrazona")
    display(region_w.rename(columns={"regiao":"Região","shift_pp":"Mudança Lula (pp)",
            "valid":"Votos tardios elegíveis","zones":"Zonas elegíveis",
            "positive":"Zonas com aumento"}).round(3))
    display(Markdown("**Para o público:** este mapa mostra onde a composição "
        "dos BUs recebidos mais tarde mudou, **não onde ocorreu fraude**. "
        "A próxima etapa é conferir o perfil das seções dessas zonas e ligar "
        "o recebimento aos lotes efetivamente publicados."))
else:
    display(Markdown("**Mapa pendente:** execute a opção W para obter a comparação "
        "por UF com controle de amostra."))

## 6 — Os votos recebidos mais tarde eram diferentes dentro das mesmas zonas?

**Pergunta:** a mudança depois das 20h04 aconteceu de repente naquele horário?

A comparação das fases V/W usa votos de BUs **recebidos** antes e depois de dez horários predefinidos. Consideramos somente zonas com pelo menos 1.000 votos válidos de cada lado de cada corte.

Atenção: **as zonas elegíveis mudam entre horários**. O aumento às 21h, por exemplo, representa um conjunto muito menor. Isso impede uma interpretação como experimento controlado.

In [ ]:
if w:
    cut=checked_csv(w["output_files"]["cutoffs"])
    cut["clock"]=pd.to_datetime(cut["cutoff_brt"])
    fig,(a1,a2)=plt.subplots(2,1,figsize=(11,7),sharex=True,
                             gridspec_kw={"height_ratios":[2,1]})
    a1.plot(cut["clock"],cut["weighted_shift_lula_pp"],color=TEAL,
            lw=2.5,marker="o",markersize=6)
    a1.axhline(0,color=GREY,lw=1,ls=":")
    a1.axvline(pd.Timestamp("2026-10-04 20:04:39"),color=RED,lw=1.5,ls="--")
    a1.set(ylabel="Lula: recebido depois − antes (pontos percentuais)",
           title="A diferença já existia antes das 20h04")
    a1.grid(axis="y",alpha=.17)
    a1.annotate("Corte investigado\n20h04min39s",
                xy=(pd.Timestamp("2026-10-04 20:04:39"),
                    float(cut.loc[cut["cutoff_brt"]=="2026-10-04 20:04:39",
                                  "weighted_shift_lula_pp"].iloc[0])),
                xytext=(16,-32),textcoords="offset points",
                color=RED,fontsize=9)
    a2.plot(cut["clock"],cut["paired_zones"],color=NAVY,
            lw=2,marker="o",markersize=5)
    a2.axvline(pd.Timestamp("2026-10-04 20:04:39"),color=RED,lw=1.5,ls="--")
    a2.set(xlabel="Horário usado para dividir os BUs por recebimento",
           ylabel="Zonas comparáveis")
    a2.grid(axis="y",alpha=.17)
    fig.autofmt_xdate()
    publish(fig,"03_mudanca_ja_acontecia_antes")
    display(Markdown("**O que observamos:** o contraste entre os BUs anteriores e "
        "posteriores ao corte já era positivo antes das 20h04. "
        "**O que isso enfraquece:** uma explicação em que toda a mudança teria "
        "começado precisamente na retomada. "
        "**O que não resolve:** se houve algum comportamento operacional adicional "
        "na publicação EA20."))
else:
    display(Markdown("Execute a **opção W** para mostrar os controles em outros horários."))

## 7 — De onde vieram as diferenças em relação à previsão por município?

As fases T/V criaram uma previsão para os BUs tardios com base nos recebidos antes de 20h04. O gráfico abaixo apresenta os **erros desse modelo em números de votos de Lula**.

**Crucial:** "votos acima do previsto" **não** quer dizer "votos falsos", "votos acrescentados" ou "votos sem BU". São erros de previsão; nosso objetivo é verificar onde ocorrem e se há uma explicação territorial.

In [ ]:
if v:
    cats=v["categories"]
    order=[
        ("MESMA_ZONA_COM_PRE","Zonas que já tinham BUs"),
        ("MUNICIPIO_SEM_PRE","Municípios sem BUs anteriores"),
        ("ZONA_SEM_PRE","Zonas novas em municípios conhecidos"),
    ]
    labels=[item[1] for item in order]
    errors=[cats[item[0]]["municipal_gap_votes"] for item in order]
    fig,ax=plt.subplots(figsize=(11,4))
    bars=ax.barh(labels,errors,color=[TEAL,TEAL,RED],height=.62)
    ax.axvline(0,color=GREY,lw=1)
    ax.invert_yaxis()
    ax.set(title="A maior parte do erro do modelo aparece em zonas já conhecidas",
           xlabel="Votos de Lula observados menos previstos (não são votos irregulares)")
    ax.grid(axis="x",alpha=.15)
    for bar,val in zip(bars,errors):
        ax.text(val+(6000 if val>=0 else -6000),
                bar.get_y()+bar.get_height()/2,
                f"{val:+,.0f}".replace(",","."),va="center",
                ha=("left" if val>=0 else "right"),fontsize=10)
    publish(fig,"04_decomposicao_do_erro_de_previsao")
    display(Markdown("**O que observamos:** o erro não se concentra "
        "exclusivamente em municípios que apareceram tarde. "
        "**Próximo teste:** distinguir a composição das seções dentro das zonas "
        "dos possíveis mecanismos de processamento dos resultados."))
else:
    display(Markdown("Execute a **opção V** para construir esta decomposição."))

## 8 — A prova operacional mais importante: recebido versus publicado

Este é o teste da opção **Y**. Para cada captura nacional EA20, comparamos quantos votos já constavam como recebidos no BWEB até aquele momento.

**Sob uma hipótese específica** — mesmo universo de votos, relógios compatíveis e recebimento anterior à publicação — o publicado não pode ultrapassar o recebido até aquele instante.

- **Diferença negativa:** o modelo simples não fecha naquele horário; precisamos investigar relógio, significado do registro ou processamento.
- **Diferença positiva:** a condição necessária foi atendida, mas não sabemos quais urnas estavam em qual lote.
- **Nunca:** declarar fraude apenas por uma diferença temporal sem certificar a equivalência dos registros.

In [ ]:
if y:
    timeline=checked_csv(y["output_files"]["timeline"])
    timeline["datetime"]=pd.to_datetime(timeline["ea20_brt"])
    pairs=[
        ("valid_received_minus_published","Votos válidos",NAVY),
        ("lula_received_minus_published","Lula",TEAL),
        ("flavio_received_minus_published","Flávio Bolsonaro",AMBER),
    ]
    fig,ax=plt.subplots(figsize=(11,5))
    for key,name,color in pairs:
        ax.plot(timeline["datetime"],
                timeline[key]/1_000_000,label=name,color=color,lw=1.9)
    ax.axhline(0,color=RED,lw=1.3,ls="--",
               label="Limite da hipótese: publicado ≤ recebido")
    ax.axvline(pd.Timestamp("2026-10-04 20:04:39"),
               lw=1.2,ls=":",color=GREY)
    ax.set(title="O que já estava recebido quando o resultado foi publicado?",
           xlabel="Horário declarado da captura EA20",
           ylabel="Recebidos menos publicados (milhões de votos)")
    ax.grid(axis="y",alpha=.15)
    ax.legend(frameon=False,ncol=2)
    fig.autofmt_xdate()
    publish(fig,"05_recebimento_vs_publicacao")
    count=y["baseline_offsets"]["snapshot_violations"]
    conclusion=("**O modelo temporal apresentou incompatibilidades em "
                +str(count)+" capturas.** Ainda é necessário conferir a "
                "semântica e sincronização dos relógios."
                if count else
                "**A condição agregada foi satisfeita nas capturas observadas.** "
                "Isso não prova que cada BU tenha sido publicado depois de seu "
                "recebimento, nem exclui outras hipóteses.")
    display(Markdown(conclusion))
else:
    display(Markdown("**Teste operacional pendente:** a opção Y ainda não "
      "produziu um resultado local verificado. Execute Y pelo menu BAT e "
      "abra novamente este painel. Não vamos preencher este gráfico "
      "com valores hipotéticos."));
    fig,ax=plt.subplots(figsize=(11,2.8))
    ax.axis("off")
    ax.text(.5,.62,"Recebido ≠ publicado",ha="center",va="center",
            fontsize=23,color=NAVY,weight="bold",transform=ax.transAxes)
    ax.text(.5,.32,"A evidência operacional será exibida após a fase Y.",
            ha="center",va="center",fontsize=12,color=GREY,transform=ax.transAxes)
    publish(fig,"05_teste_operacional_pendente")

## 9 — Antes do segundo turno, alguns modelos diziam que Lula quase não tinha chance. E então?

**Este é um novo contexto eleitoral, separado dos dados de recebimento/publicação do PRIMEIRO turno analisados nas seções anteriores.**

Um conjunto de projeções do segundo turno foi registrado no GitHub em **4 de outubro às 23h44 (Brasília)**, semanas antes da votação final. O método estrutural **M1** atribuía a Lula apenas **0,3% de chances de vencer**. O mesmo autor, porém, publicou dois métodos alternativos com probabilidades de **31,7%** e **41,1%** e uma combinação ponderada de **17,9%**.

Em 24 de outubro, outras duas estimativas publicadas davam **16%** (Plano Político) e **17%** (Opinião). O segundo turno terminou no dia 25 de outubro com **51,79% de Lula e 48,21% de Flávio Bolsonaro**.

**Detalhe que evita um erro de interpretação:** outra fonte tinha um número de *2%* para o cenário de **Lula terminar com mais de 52% dos votos**, diferente da chance de simplesmente vencer. Como Lula terminou com 51,79%, aquele cenário específico nem aconteceu.

Uma previsão estatística pode errar por incerteza subestimada, pesquisas enviesadas, premissas de transferência de votos ou mudanças reais de preferência/comparecimento. **A chance estimada de vitória NÃO é probabilidade de fraude.** O que precisamos auditar em paralelo continua sendo a correspondência de cada total com seu BU.

Fontes com data e método: [projeção M1/M2/M3 arquivada em 05/10](https://github.com/berndof/eleicao-2026/blob/528b1d8b04c999e3c7833c181c564fd1e5e299c3/docs/artigo.pt.md), [Plano Político, 24/10](https://www.planopolitico.com.br/modelos-eleitorais/presidencial/), [Opinião, 24/10](https://eleicoes2026.opiniao.inf.br/analises/segundo-turno/) e [resultado oficial publicado pelo TSE](https://www.tse.jus.br/comunicacao/noticias/2026/Outubro/presidente-do-tse-anuncia-reeleicao-de-lula-e-faz-balanco-do-2o-turno-das-eleicoes-2026).

In [ ]:
# Contexto eleitoral separado: NÃO carregar nem misturar dados BWEB/EA20 do 1º turno.
poll_source=REPO/"data/forensics/external/segundo_turno_previsoes_2026_proveniencia.json"
ctx=json.loads(poll_source.read_text(encoding="utf-8"))
assert ctx["version"]=="contexto-previsoes-segundo-turno-2026-v1"
variants=ctx["first_pre_runoff_archive"]["model_variants"]
assert len(variants)==4
official=ctx["official_runoff_result"]
assert official["date"]=="2026-10-25"
assert official["lula_votes"]+official["flavio_votes"]==official["valid_votes"]
fig,ax=plt.subplots(figsize=(11,5))
prob=[z["lula_win_probability_pct"] for z in variants]
labels=[z["name"] for z in variants]
colors=[TEAL,AMBER,NAVY,GREY]
ax.barh(labels,prob,color=colors,height=.64)
ax.invert_yaxis()
ax.set_xlim(0,100)
ax.set_xlabel("% das simulações daquele modelo em que Lula venceria")
ax.set_title("Uma previsão de 0,3% não representava o consenso dos modelos",loc="left")
for i,p in enumerate(prob):
    ax.text(p+1,i,("%.1f%%"%p).replace(".",","),va="center",fontsize=11,weight="bold")
ax.text(.02,-.25,"Previsão registrada antes do 2º turno • 04/10",
        fontsize=9,color=GREY,transform=ax.transAxes)
ax.grid(axis="x",alpha=.12)
publish(fig,"09_probabilidades_modelos_2turno")

fig,ax=plt.subplots(figsize=(10.5,5))
for i,entry in enumerate(variants):
    low,high=entry["interval_90_pct"]
    mean=entry["lula_valid_share_mean_pct"]
    ax.plot([low,high],[i,i],lw=5,color=colors[i],alpha=.7,solid_capstyle="round")
    ax.plot([mean],[i],marker="o",markersize=9,color=colors[i])
    ax.text(high+.22,i,f"{mean:.2f}% previsto".replace(".",","),va="center",
            fontsize=9,color=GREY)
actual=official["lula_valid_pct"]
ax.axvline(actual,color=RED,linestyle="--",lw=2,
           label="Resultado oficial: Lula 51,79%")
ax.set_yticks(range(len(labels)),labels)
ax.invert_yaxis()
ax.set_xlim(40,61)
ax.set_xlabel("Participação de Lula nos votos válidos do 2º turno (%)")
ax.set_title("O resultado ficou fora da faixa central do M1,
mas dentro da faixa de outros modelos",loc="left")
ax.grid(axis="x",alpha=.12)
ax.legend(frameon=False,loc="lower right")
publish(fig,"10_projecoes_versus_resultado_2turno")

comparison=pd.DataFrame([
    {"Fonte": "M1 estrutural (04/10)", "Lula vence (% estimado)":0.3},
    {"Fonte": "M2 pesquisas corrigidas (04/10)","Lula vence (% estimado)":31.7},
    {"Fonte": "M3 histórico (04/10)","Lula vence (% estimado)":41.1},
    {"Fonte": "Combinação M1/M2/M3 (04/10)","Lula vence (% estimado)":17.9},
]+[{"Fonte":z["provider"]+" ("+z["date"][8:10]+"/10)",
     "Lula vence (% estimado)":z["lula_win_probability_pct"]}
    for z in ctx["other_pre_runoff_models"] if z["date"]=="2026-10-24"])
display(comparison)
display(Markdown("**Conclusão:** o resultado contrariou fortemente a previsão "
    "do M1, cujo próprio autor reconhecia possível incerteza subestimada. "
    "Outros modelos já atribuíam probabilidades substanciais à vitória de Lula. "
    "Não existe aqui um teste de adulteração de boletins de urna; isso exige "
    "reconciliação eleitoral independente dos modelos de pesquisa."))

## 10 — Quais hipóteses resistiram e o que ainda não demonstramos?

O papel do investigador não é colecionar anomalias. É **reduzir as explicações possíveis**, documentar refutações e indicar o teste que mudaria nossa conclusão. Esta é a situação atual da investigação, não um veredicto político.

In [ ]:
findings=[
    ("A linha acumulada é muito lisa",
     "Observado na série EA20",
     "Pode surgir de acumulação + sequência de volumes; oráculo é circular",
     "Não demonstra adulteração"),
    ("Os votos tardios têm composição distinta",
     "Observado em EA20 e BWEB, por métricas diferentes",
     "Geografia, seções e horário de recepção influenciam",
     "Ainda falta vincular BU a lote publicado"),
    ("O gradiente começou precisamente às 20h04",
     "Não sustentado pelos cortes W",
     "Já havia diferença antes, embora coortes mudem",
     "Não elimina outros eventos operacionais"),
    ("Votos divulgados sem terem sido recebidos",
     "Teste Y realizado" if y else "Pendente de execução da fase Y",
     "Exige relógios e conceitos compatíveis",
     "Sem identidade de seção, não é prova de alteração"),
    ("BU individual diverge do resultado oficial",
     "Não demonstrado por estas fases de curva",
     "Requer auditoria por seção e validação de autenticidade",
     "Sem conclusão de fraude a partir destas curvas"),
]
df=pd.DataFrame(findings,columns=[
    "Pergunta","Situação nos dados","Explicação/controle","Limite da conclusão"])
display(df)
display(Markdown(
    "**Conclusão provisória:** existem padrões de recebimento e divulgação "
    "que merecem investigação operacional e documentação pública. "
    "Os resultados aqui examinados **não constituem demonstração de fraude "
    "eleitoral**. Para avançar, nossa prioridade é o confronto por seção "
    "entre o BU verificável e o que foi efetivamente publicado, ou o acesso "
    "a registros temporais que permitam esse vínculo."))
print("Figuras para divulgação (PNG):",EXPORT)
print("Apenas resultados locais certificados foram utilizados.");


## 11 — Como divulgar sem induzir uma conclusão que os dados não sustentam

### Roteiro popular sugerido

> **Estamos auditando os números da apuração de 2026.** Descobrimos que a curva de divulgação era muito regular, embora os grupos de votos divulgados tivessem perfis diferentes. A diferença entre votos mais cedo e mais tarde já aparecia antes das 20h04. Isso não comprova fraude — nem dispensa investigação. O próximo passo é conferir se o que o sistema publicou era compatível com o que as urnas registraram e quando esses registros foram recebidos.

### Legenda que deve acompanhar qualquer imagem

**Fonte:** projeto TSE, primeiro turno presidencial de 2026, cálculos reproduzíveis sobre BWEB e capturas EA20. A visualização é descritiva; não identifica alteração de votos por si só. Consulte os scripts e hashes dos resultados da fase citada.

As figuras em PNG ficam salvas na pasta `H:\TSE\forensics\urnas\analysis\periodicidade_bweb_2026\divulgacao_publica_2026`.

**Próxima etapa editorial:** substituir qualquer mensagem de "fraude provada" por uma cadeia de prova verificável e publicar lado a lado os dados, o gráfico, a explicação alternativa e o documento que poderia decidir a controvérsia.

## 12 — Criar uma página para qualquer pessoa ler sem instalar o Jupyter

A célula abaixo monta um **relatório HTML independente**, com os mesmos gráficos, legendas em português e identificação explícita do que foi demonstrado e do que continua inconclusivo.

Esse é o formato indicado para compartilhar os achados com o público. Ele preserva as imagens dentro do próprio HTML; basta copiar o arquivo gerado. **Revise a redação e a integridade dos números antes de publicá-lo.**

In [ ]:
import base64
from html import escape

def image_in_html(png,caption):
    path=EXPORT/png
    if not path.is_file():
        return ""
    encoded=base64.b64encode(path.read_bytes()).decode("ascii")
    return ('<figure><img alt="'+escape(caption)+'" src="data:image/png;base64,'
            +encoded+'"><figcaption>'+escape(caption)+'</figcaption></figure>')

parts=[
    '<!doctype html><html lang="pt-BR"><head><meta charset="utf-8">',
    '<meta name="viewport" content="width=device-width,initial-scale=1">',
    '<title>Investigação forense da apuração 2026 — O que os dados mostram?</title>',
    '<style>body{font-family:system-ui,-apple-system,Segoe UI,sans-serif;'
    'max-width:900px;margin:40px auto;padding:0 22px;line-height:1.65;color:#183153}'
    'h1{line-height:1.2;font-size:2rem}h2{margin-top:42px;border-top:1px solid #d6dce2;padding-top:22px}'
    'img{width:100%;height:auto;border:1px solid #e5e7eb;border-radius:8px}'
    'figure{margin:22px 0}figcaption,.note{color:#5d6571;font-size:.9rem}'
    'strong{font-weight:700}a{color:#197a74}footer{margin-top:48px;font-size:.9rem}'
    '</style></head><body>',
    '<h1>Investigação forense da apuração de 2026</h1>',
    '<p><strong>Pergunta:</strong> há evidências verificáveis de votos alterados,'
    ' acrescentados ou omitidos? Os resultados são apresentados com controles'
    ' alternativos e limites explícitos — não se confunde anomalia visual com fraude.</p>',
]
if x:
    parts.extend([
        '<h2>1. A linha acumulada</h2>',
        '<p>O gráfico de divulgação é bastante regular. Uma simulação que mantém'
        ' os volumes reais de cada lote, mas conhece a proporção final de votos,'
        ' pode ser ainda mais regular. Esse modelo é retrospectivo e não prova a'
        ' existência de um mecanismo específico.</p>',
        image_in_html("01_linha_acumulada_e_controle",
            "Acumulado real EA20 e controle de composição constante; eixo vertical ampliado"),
        '<h2>2. Os novos votos não vieram em lotes iguais</h2>',
        '<p>Entre os quatro blocos de votos publicados, a composição mudou.'
        ' A média acumulada pode ocultar mudanças de composição e tamanhos de lote.</p>',
        image_in_html("02_quatro_blocos_de_votos",
            "Composição Lula, Flávio e demais votos válidos nos quatro blocos EA20"),
    ])
if t:
    parts.extend([
        '<h2>3. Onde os votos foram recebidos mais tarde?</h2>',
        '<p>O mapa colore estados pela fração dos próprios votos válidos'
        ' BWEB recebidos a partir de 20h04. Não é um mapa de fraude nem'
        ' da publicação EA20; ZZ (exterior) fica fora do território brasileiro.</p>',
        image_in_html("06_mapa_brasil_recebimento_tardio",
            "Brasil por UF: % dos votos da UF recebidos depois das 20h04"),
    ])
if w:
    parts.extend([
        '<h2>4. Mudança de composição dentro das zonas: estados e regiões</h2>',
        '<p>Nas zonas com no mínimo mil votos válidos antes e depois do corte,'
        ' calculamos quanto mudou a participação de Lula entre os BUs tardios.'
        ' Diferença estatística não equivale a adulteração de votos.</p>',
        image_in_html("07_mapa_brasil_mudanca_intrazona",
            "Brasil por UF: Lula pós menos pré, apenas zonas comparáveis"),
        image_in_html("08_regioes_brasil_mudanca_intrazona",
            "Regiões brasileiras: média ponderada nas zonas comparáveis"),
    ])
if w:
    parts.extend([
        '<h2>5. A diferença temporal dentro das zonas</h2>',
        '<p>Comparações de BUs recebidos antes e depois de vários horários mostram'
        ' diferença anterior às 20h04. As zonas elegíveis mudam com o horário;'
        ' não é um ensaio aleatorizado nem medida de fraude.</p>',
        image_in_html("03_mudanca_ja_acontecia_antes",
            "Deslocamento percentual e número de zonas comparáveis em cortes horários BWEB"),
    ])
if v:
    parts.extend([
        '<h2>6. Onde o modelo de previsão errou?</h2>',
        '<p>Votos acima ou abaixo de uma previsão são erros de previsão,'
        ' <strong>não votos irregulares</strong>. A decomposição permite'
        ' direcionar a investigação para as zonas já observadas.</p>',
        image_in_html("04_decomposicao_do_erro_de_previsao",
            "Diferenças de Lula observado menos previsto nas classes de zonas"),
    ])
parts.append('<h2>7. A sequência operacional: receber e publicar</h2>')
if y:
    count=y["baseline_offsets"]["snapshot_violations"]
    parts.append('<p>Nas '+str(y["ea20_snapshots"])+' capturas EA20, a condição'
        ' temporal conjunta apresentou '+str(count)+' incompatibilidades.'
        ' Esse número testa hipóteses sobre relógios e processamento;'
        ' não identifica, isoladamente, adulteração de votos.</p>')
    parts.append(image_in_html("05_recebimento_vs_publicacao",
        "Diferença entre recebidos acumulados BWEB e publicados EA20"))
else:
    parts.append('<p><strong>Teste pendente.</strong> O resultado da fase Y'
        ' não estava disponível ao gerar esta página.'
        ' A ausência deste dado não deve ser preenchida por conjecturas.</p>')
parts.extend([
    '<h2>8. As simulações de antes do segundo turno</h2>',
    '<p><strong>Atenção: contexto separado do primeiro turno.</strong>'
    ' O modelo estrutural M1 publicado em 04/10 atribuía 0,3% de chance'
    ' a uma vitória de Lula, mas a combinação de três modelos do mesmo autor'
    ' apontava 17,9%. Em 24/10, outras estimativas publicadas'
    ' indicavam 16% e 17%. O resultado oficial de 25/10 foi Lula 51,79%.'
    ' Nenhuma dessas probabilidades estima chance de fraude.</p>',
    image_in_html("09_probabilidades_modelos_2turno",
        "Probabilidade de vitória de Lula em quatro métodos antes do segundo turno"),
    image_in_html("10_projecoes_versus_resultado_2turno",
        "Faixas de projeção dos votos válidos no segundo turno e resultado do TSE"),
    '<p class="note">Fontes: <a href="https://github.com/berndof/eleicao-2026/'
    'blob/528b1d8b04c999e3c7833c181c564fd1e5e299c3/docs/artigo.pt.md">'
    'projeção arquivada (05/10)</a>;'
    ' <a href="https://www.planopolitico.com.br/modelos-eleitorais/presidencial/">'
    'Plano Político</a>;'
    ' <a href="https://eleicoes2026.opiniao.inf.br/analises/segundo-turno/">Opinião</a>;'
    ' <a href="https://www.tse.jus.br/comunicacao/noticias/2026/Outubro/'
    'presidente-do-tse-anuncia-reeleicao-de-lula-e-faz-balanco-do-2o-turno-das-eleicoes-2026">'
    'resultado oficial TSE</a>.'
    ' Esses modelos têm premissas e datas diferentes e não auditam BUs.</p>',
])
parts.extend([
    '<h2>O que esta investigação pode concluir?</h2>',
    '<p><strong>Observado:</strong> a curva acumulada é regular; os lotes'
    ' publicados e recebidos têm composições e ritmos diferentes.'
    ' <strong>Não demonstrado:</strong> alteração de votos por esses gráficos.'
    ' <strong>Para decidir a hipótese de fraude:</strong> verificar BUs'
    ' autenticados por seção, contagens publicadas correspondentes e'
    ' registros temporais com semântica e relógios certificados.</p>',
    '<footer><p><strong>Fonte:</strong> projeto independente TSE — primeiro turno'
    ' presidencial de 4 de outubro de 2026. Repositório: '
    '<a href="https://github.com/rubentrancoso/TSE">github.com/rubentrancoso/TSE</a>.'
    ' Todos os gráficos são derivados de saídas locais verificadas por SHA-256.'
    ' A análise não tem endosso institucional e não é uma conclusão judicial'
    ' ou pericial sobre fraude.</p></footer></body></html>'
])
html_path=EXPORT/"relatorio_publico_forense_2026.html"
html_path.write_text("\n".join(parts),encoding="utf-8")
print("Relatório público autônomo:",html_path)
print("O HTML inclui as figuras; pode ser aberto sem Jupyter.")